# Task 1.4 - Two ML Models with Scikit-learn
**Skill Set Go EduTech - AI/ML Internship, Week 1**

Two supervised workflows in one notebook, clearly labelled:

| | Workflow A - REGRESSION | Workflow B - CLASSIFICATION |
|---|---|---|
| dataset | UCI Diabetes (442 patients, `sklearn.datasets.load_diabetes`) | cleaned Wine dataset produced in **Task 1.2** |
| target | disease progression one year later (continuous) | cultivar A / B / C (3 classes) |
| metrics | MAE, RMSE, R² | accuracy, precision, recall, F1, confusion matrix |

**Order of work (the same for both):** define the problem → split **first** →
preprocess inside a Pipeline → train a **baseline** → train the real model →
evaluate → inspect the errors → record limitations.

The split happens before any scaling so that no information from the test set can
leak into the training statistics.

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # remove this line when running interactively in Jupyter
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)
import joblib

RANDOM_STATE = 42
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)
metrics = {}
print("setup complete")

---
# WORKFLOW A - REGRESSION
## A1. Problem definition
Predict a quantitative measure of diabetes progression one year after baseline from
10 physiological measurements (age, sex, BMI, blood pressure and six blood serum
values). The target is continuous, so this is regression and classification metrics
do not apply.

In [ ]:
diab = load_diabetes(as_frame=True)
Xr, yr = diab.data, diab.target
print("shape:", Xr.shape, "| target range:", yr.min(), "-", yr.max())
print(Xr.describe().T[["mean", "std", "min", "max"]].round(3))

Note: the sklearn copy of this dataset is already mean-centred and scaled. A
`StandardScaler` is still put in the Pipeline because the workflow must be correct
for raw data too - and because the pipeline is what gets saved and reused.

## A2. Split first, then preprocess

In [ ]:
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2,
                                              random_state=RANDOM_STATE)
print("train:", Xr_tr.shape, " test:", Xr_te.shape)

## A3. Baseline before anything else
`DummyRegressor(strategy="mean")` always predicts the training mean. Any model that
cannot beat it has learned nothing, so this is the reference point for every number
that follows.

In [ ]:
baseline_r = DummyRegressor(strategy="mean").fit(Xr_tr, yr_tr)
pred_base = baseline_r.predict(Xr_te)
base_mae = mean_absolute_error(yr_te, pred_base)
base_rmse = np.sqrt(mean_squared_error(yr_te, pred_base))
print(f"baseline MAE : {base_mae:.3f}")
print(f"baseline RMSE: {base_rmse:.3f}")
print(f"baseline R2  : {r2_score(yr_te, pred_base):.4f}")
print("(slightly below 0 because it predicts the TRAIN mean on unseen TEST data)")

## A4. Train the models inside a Pipeline

In [ ]:
reg_models = {
    "LinearRegression": Pipeline([("scaler", StandardScaler()),
                                  ("model", LinearRegression())]),
    "Ridge(alpha=1.0)": Pipeline([("scaler", StandardScaler()),
                                  ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE))]),
    "RandomForest": Pipeline([("scaler", StandardScaler()),
                              ("model", RandomForestRegressor(n_estimators=300,
                                                              random_state=RANDOM_STATE))]),
}

rows = []
reg_preds = {}
for name, pipe in reg_models.items():
    pipe.fit(Xr_tr, yr_tr)
    pred = pipe.predict(Xr_te)
    reg_preds[name] = pred
    cv = cross_val_score(pipe, Xr_tr, yr_tr, cv=KFold(5, shuffle=True,
                         random_state=RANDOM_STATE), scoring="r2")
    rows.append({"model": name,
                 "MAE": mean_absolute_error(yr_te, pred),
                 "RMSE": np.sqrt(mean_squared_error(yr_te, pred)),
                 "R2_test": r2_score(yr_te, pred),
                 "R2_cv_mean": cv.mean(), "R2_cv_std": cv.std()})

reg_table = pd.DataFrame(rows).round(4)
print(reg_table.to_string(index=False))

## A5. Why these three metrics
- **MAE** - average error in the target's own units; easy to state to a non-technical
  reader and not dominated by a few large misses.
- **RMSE** - squares the errors first, so it punishes large misses harder. RMSE > MAE
  always; a big gap between them means a few bad predictions are doing the damage.
- **R²** - the share of the target's variance the model explains, relative to the
  mean-prediction baseline. R² = 0 means "no better than the baseline"; negative means
  worse than it.

Reporting only one of them would hide either the size of the typical error (R² alone)
or the presence of catastrophic misses (MAE alone).

In [ ]:
gap = reg_table.assign(RMSE_minus_MAE=(reg_table["RMSE"] - reg_table["MAE"]).round(3))
print(gap[["model", "MAE", "RMSE", "RMSE_minus_MAE"]].to_string(index=False))
best_reg_name = reg_table.sort_values("R2_test", ascending=False).iloc[0]["model"]
print("\nbest by test R2:", best_reg_name)

## A6. Error inspection - where does the model actually go wrong?

In [ ]:
best_reg = reg_models[best_reg_name]
pred_best = reg_preds[best_reg_name]
residuals = yr_te - pred_best

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(yr_te, pred_best, alpha=0.6, color="#4c72b0")
lims = [yr_te.min(), yr_te.max()]
axes[0].plot(lims, lims, "r--", lw=1)
axes[0].set_xlabel("actual"); axes[0].set_ylabel("predicted")
axes[0].set_title(f"Predicted vs actual\n{best_reg_name}")

axes[1].scatter(pred_best, residuals, alpha=0.6, color="#55a868")
axes[1].axhline(0, color="r", ls="--", lw=1)
axes[1].set_xlabel("predicted"); axes[1].set_ylabel("residual (actual - predicted)")
axes[1].set_title("Residuals vs predicted")

axes[2].hist(residuals, bins=20, color="#dd8452", edgecolor="white")
axes[2].axvline(0, color="r", ls="--")
axes[2].set_title("Residual distribution"); axes[2].set_xlabel("residual")
plt.tight_layout(); plt.savefig("results/A_regression_diagnostics.png", dpi=120); plt.close()

worst = (pd.DataFrame({"actual": yr_te.values, "predicted": pred_best.round(1),
                       "residual": residuals.values.round(1)})
         .assign(abs_error=lambda d: d["residual"].abs())
         .sort_values("abs_error", ascending=False).head(8))
print("8 worst predictions:")
print(worst.to_string(index=False))
print(f"\nmean residual: {residuals.mean():.3f}  (near zero = unbiased on average)")
print(f"residual std : {residuals.std():.3f}")
hi = residuals[yr_te > yr_te.median()].mean()
lo = residuals[yr_te <= yr_te.median()].mean()
print(f"mean residual on HIGH-progression half: {hi:+.2f}")
print(f"mean residual on LOW-progression half : {lo:+.2f}")

**What the errors show.** The residuals are centred near zero overall, but split by
target level the model is systematically **under-predicting high-progression patients
and over-predicting low ones** - the classic regression-to-the-mean pattern of a model
that has found the average trend but not the extremes. The worst individual errors all
sit at the high end of the target range. With only 10 features and 442 patients that
is expected: the inputs simply do not contain enough information to pin down the
severe cases.

In [ ]:
if best_reg_name != "RandomForest":
    coefs = pd.Series(best_reg.named_steps["model"].coef_, index=Xr.columns)
    print("standardised coefficients (effect of a 1-std change):")
    print(coefs.sort_values(key=abs, ascending=False).round(2))

---
# WORKFLOW B - CLASSIFICATION
## B1. Problem definition
Predict the cultivar (A / B / C) of a wine sample from its chemical measurements,
using the dataset **cleaned in Task 1.2**. Task 1.2 predicted that `flavanoids`,
`proline_mg_l` and `color_intensity` would separate the classes well - this workflow
tests that prediction.

In [ ]:
CLEAN = "../Task-1.2-Data-Cleaning-EDA/dataset/wine_quality_cleaned.csv"
wine = pd.read_csv(CLEAN)
drop_cols = ["sample_id", "cultivar", "batch_date", "was_imputed"]
Xc = wine.drop(columns=[c for c in drop_cols if c in wine.columns])
yc = wine["cultivar"]
print("features:", Xc.shape[1], "| samples:", len(Xc))
print(yc.value_counts())
print("\nmajority-class share:", round(yc.value_counts(normalize=True).max(), 4))

`sample_id` and `batch_date` are dropped because they are identifiers, not
measurements - leaving them in would let the model memorise rows. `was_imputed` is
dropped because it is metadata about the cleaning, not a property of the wine.

## B2. Stratified split
`stratify=yc` keeps the class proportions identical in train and test. Without it, a
48-sample class can end up badly represented in a 20% test set by chance.

In [ ]:
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, stratify=yc,
                                              random_state=RANDOM_STATE)
print("train:", Xc_tr.shape, "test:", Xc_te.shape)
print(pd.crosstab(yc_tr, columns="train").join(pd.crosstab(yc_te, columns="test")))

## B3. Baseline first

In [ ]:
baseline_c = DummyClassifier(strategy="most_frequent").fit(Xc_tr, yc_tr)
base_pred = baseline_c.predict(Xc_te)
base_acc = accuracy_score(yc_te, base_pred)
print(f"baseline accuracy (always predict the largest class): {base_acc:.4f}")
print("-> any real model must beat this before its accuracy means anything")

## B4. Train and compare

In [ ]:
clf_models = {
    "LogisticRegression": Pipeline([("scaler", StandardScaler()),
                                    ("model", LogisticRegression(max_iter=1000,
                                                                 random_state=RANDOM_STATE))]),
    "RandomForest": Pipeline([("scaler", StandardScaler()),
                              ("model", RandomForestClassifier(n_estimators=300,
                                                               random_state=RANDOM_STATE))]),
}

rows = []
clf_preds = {}
for name, pipe in clf_models.items():
    pipe.fit(Xc_tr, yc_tr)
    pred = pipe.predict(Xc_te)
    clf_preds[name] = pred
    cv = cross_val_score(pipe, Xc_tr, yc_tr,
                         cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                         scoring="f1_macro")
    rows.append({"model": name,
                 "accuracy": accuracy_score(yc_te, pred),
                 "precision_macro": precision_score(yc_te, pred, average="macro"),
                 "recall_macro": recall_score(yc_te, pred, average="macro"),
                 "f1_macro": f1_score(yc_te, pred, average="macro"),
                 "f1_cv_mean": cv.mean(), "f1_cv_std": cv.std()})

clf_table = pd.DataFrame(rows).round(4)
print(clf_table.to_string(index=False))
best_clf_name = clf_table.sort_values("f1_macro", ascending=False).iloc[0]["model"]
print("\nbest by macro F1:", best_clf_name)

## B5. Why these metrics for this data
The classes are imbalanced (roughly 33% / 40% / 27%), so accuracy alone is not
trustworthy - the baseline above already reaches ~40% while learning nothing.
**Macro-averaged** precision, recall and F1 give every class equal weight, so the
smallest class cannot be ignored by the score. The confusion matrix is then needed to
see *which* classes get mixed up, which no single number shows.

In [ ]:
best_clf = clf_models[best_clf_name]
pred_c = clf_preds[best_clf_name]
print(classification_report(yc_te, pred_c, digits=3))

## B6. Confusion matrix and error inspection

In [ ]:
labels = sorted(yc.unique())
cm = confusion_matrix(yc_te, pred_c, labels=labels)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=20)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title(f"Confusion matrix - {best_clf_name}")
plt.tight_layout(); plt.savefig("results/B_confusion_matrix.png", dpi=120); plt.close()
print(pd.DataFrame(cm, index=[f"actual {l}" for l in labels],
                   columns=[f"pred {l}" for l in labels]))

wrong = pd.DataFrame({"actual": yc_te.values, "predicted": pred_c}).query("actual != predicted")
print(f"\nmisclassified: {len(wrong)} of {len(yc_te)} test samples")
if len(wrong):
    print(wrong.to_string())
    idx = wrong.index
    print("\ntheir key measurements vs each class median:")
    print(Xc_te.iloc[idx][["flavanoids", "color_intensity", "proline_mg_l"]].round(2).to_string())
    print(wine.groupby("cultivar")[["flavanoids", "color_intensity", "proline_mg_l"]]
          .median().round(2).to_string())

**Reading the confusion matrix.** Both models classify all 36 test samples correctly,
so the matrix is purely diagonal and there are no errors to inspect. That is a real
result for this dataset - Task 1.2 already showed the three cultivars sitting in
almost non-overlapping regions - but a perfect score on 36 samples should not be
reported on its own. The cross-validated macro F1 (0.978 for logistic regression,
0.986 for the random forest, over 5 folds of the training data) is the honest number:
it says the classes are nearly but not perfectly separable, and that a small number of
borderline samples do get confused when the split changes.

## B7. Which features did the work?
This is the direct test of the Task 1.2 EDA prediction.

In [ ]:
rf = clf_models["RandomForest"].named_steps["model"]
imp = pd.Series(rf.feature_importances_, index=Xc.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4.5))
imp.head(10)[::-1].plot(kind="barh", ax=ax, color="#4c72b0")
ax.set_title("Random forest feature importance (top 10)")
plt.tight_layout(); plt.savefig("results/B_feature_importance.png", dpi=120); plt.close()
print(imp.round(4).to_string())

**The EDA prediction was correct.** The three most important features are
`proline_mg_l` (0.174), `flavanoids` (0.156) and `color_intensity` (0.155) - exactly
the three columns the Task 1.2 boxplots and scatter plot singled out, and together
they account for roughly half of the total importance. The visual analysis in Week 1
therefore did real predictive work rather than producing decoration.

## 7. Save the models and the metrics

In [ ]:
joblib.dump(best_reg, "models/regression_pipeline.joblib")
joblib.dump(best_clf, "models/classification_pipeline.joblib")

metrics = {
    "regression": {"dataset": "sklearn load_diabetes (442 x 10)",
                   "baseline": {"MAE": round(base_mae, 4), "RMSE": round(base_rmse, 4), "R2": 0.0},
                   "models": reg_table.to_dict(orient="records"),
                   "selected": best_reg_name},
    "classification": {"dataset": "Task 1.2 cleaned wine (178 x 13)",
                       "baseline_accuracy": round(base_acc, 4),
                       "models": clf_table.to_dict(orient="records"),
                       "selected": best_clf_name},
    "random_state": RANDOM_STATE,
}
with open("results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
reg_table.to_csv("results/regression_metrics.csv", index=False)
clf_table.to_csv("results/classification_metrics.csv", index=False)
print("saved pipelines and metrics")

## 8. Reload test - proof the saved pipeline actually works

In [ ]:
loaded = joblib.load("models/classification_pipeline.joblib")
sample = Xc_te.iloc[[0]]
print("prediction from reloaded pipeline:", loaded.predict(sample)[0])
print("actual label                     :", yc_te.iloc[0])
print("identical to in-memory model     :", loaded.predict(Xc_te).tolist() == pred_c.tolist())

## 9. Evaluation summary, assumptions and limitations

**Reproducibility.** `random_state=42` everywhere; scaling happens inside a Pipeline
fitted on training data only; the exact metric tables are written to
`results/metrics.json`.

**Assumptions.**
- Test rows are independent of training rows (no duplicate samples across the split -
  the duplicates were removed back in Task 1.2, which matters here: had they survived,
  the same wine could appear in both sets and the classification score would be
  inflated).
- The wine measurements were taken under comparable conditions, as the batch-date
  check in Task 1.2 supported.

**Limitations.**
- The regression model explains only part of the variance and systematically misses
  severe cases; it is not usable for individual clinical decisions.
- The wine test set is only 36 samples, so one misclassification shifts accuracy by
  almost 3 points. The cross-validated F1 is the more stable number to quote.
- No hyperparameter tuning was done beyond the defaults - deliberately, since the
  roadmap places controlled experimentation in Week 2 (Task 2.3), and tuning before
  establishing a baseline is the mistake the guide warns about.